<!-- colab-badge -->
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RemusTeodorescu/DL-for-Engineers-Public-Course/blob/main/exercises/Ex05-cnn-and-gnn/Ex05_03_gnn_six_bus_network.ipynb)

*Open this notebook in Google Colab. Its first code cell fetches the set's library files from the public course repository, so nothing needs uploading.*

<!-- course-header v3 -->

**Deep Learning for Engineering** · MSc, Aalborg University · 2026

Developed by **Remus Teodorescu** (ret@et.aau.dk), with support from Research Assistant **Noman Khan** (nomank@energy.aau.dk).

*Reference texts — read for the theory. Used as an **inspirational source** for this course, not as a source of its code:*

- Prince, *Understanding Deep Learning*, MIT Press 2023.
- Goodfellow, Bengio & Courville, *Deep Learning*, MIT Press 2016.  Ch. 10 for recurrence and the LSTM.

Every notebook in this course has been **written and rewritten by the authors named above**. The code, the problems, the data and the exposition are **original to this course** and are not derived from any publisher's code listings or companion notebooks. Where notation matches a textbook's it is the standard notation of the field, and where an idea is a named author's it is cited as theirs in the text.

See `docs/PROVENANCE.md` for what each reference is cited for, set by set.

---

# Ex_05 · Notebook 03 — Node Regression on a Six-Bus Network

**Deep Learning for Engineering · Aalborg University · Part 1**

This is the notebook the exercise set is built around. **L12.1 slide 12 says
"you met graph neural networks in Part 1". This is where.**

The task is *node regression*: given what is injected at every bus, predict the
electrical state at every bus. Six input vectors in, six output vectors out, on
a fixed network of eight lines. The buses are the same six buses **Ex_12.1**
uses — there with real Energinet injections and a physics-informed estimator,
here with a plain supervised regression on synthetic data.

Four things happen here.

1. You read what part of the dataset is real physics and what part is a
   teaching simplification.
2. You build a message-passing layer and train a small graph network.
3. You sweep the depth and find that the answer is the graph's diameter.
4. You demonstrate permutation equivariance numerically, on the graph network
   and on a dense baseline, and watch one of them survive.

---

## 0 · Setup


In [ ]:
# files-cell v1 ----------------------------------------------------------
# This set's library files must sit beside the notebook. Locally they
# already do. On Google Colab, where a notebook opens on its own, they are
# fetched from the public course repository. Run this cell first.
import os, urllib.request
FILES = ['Ex_5_core.py']
URL = "https://raw.githubusercontent.com/RemusTeodorescu/DL-for-Engineers-Public-Course/main/exercises/Ex05-cnn-and-gnn/"
for f in FILES:
    if not os.path.exists(f):
        urllib.request.urlretrieve(URL + f, f)
        print("fetched", f)
print("files ready:", ", ".join(FILES))


In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

import Ex_5_core as core

np.set_printoptions(precision=4, suppress=True)
core.set_seed(0)

data = core.six_bus_dataset(n_cases=800, seed=12)
X, Y, A = data["X"], data["Y"], data["A"]
scale = data["Y_scale"]

print("features X :", X.shape, " ", list(data["feature_names"]))
print("targets  Y :", Y.shape, " ", list(data["target_names"]))
print("adjacency A:", A.shape)
print()
print("target standard deviation per channel:", np.round(scale, 5))
print("measurement noise                    :", data["noise"])

core.plot_graph(node_values=data["P"][0],
                title="One operating point: active injection at each bus [p.u.]")
plt.show()

**What you should see.** `features X : (800, 6, 5)` with names
`['P', 'Q', 'is_gen', 'is_load', 'is_reference']`, `targets Y : (800, 6, 2)` with
names `['theta [rad]', 'V - 1 [p.u.]']`, target standard deviations of about
`[0.144  0.0199]`, a noise level of `0.0005`, and the network drawn with
generating buses red and load buses blue.

`(800, 6, 5)` is `(cases, nodes, features)` — not a flattened `(800, 30)`
matrix. The nodes axis is kept separate so that the model can be forbidden from
caring about the order of it.

---

## 1 · What is real and what is not

Every applied model mixes measured and assumed quantities; the discipline is
labelling them.

**Real physics.** The topology; the susceptance matrix $B$ (a weighted graph
Laplacian, singular by construction); the sign conventions; the requirement
that active injections sum to zero. The bus angles solve

$$P_i \;=\; \sum_{j\in\mathcal{N}(i)} b_{ij}\,\sin(\theta_i - \theta_j)$$

by Newton-Raphson — the lossless AC power flow. Drop the sine and you get the
linearised "DC" power flow, which section 5 uses as a baseline.

**A teaching simplification.** Everything about voltage magnitude. A real AC
power flow solves magnitudes and angles together, with resistances and reactive
limits. Here $V$ is one per unit plus a linear response to reactive injection
minus a sag proportional to squared line angle differences. Right shape, wrong
numbers.

**Also invented.** The line susceptances, the injection ranges, the noise
level, and bus 5 being an HVDC infeed.

The map from injections to state is nonlinear, local, and defined on a graph.
Those three properties are what the architecture exploits, and all three
survive the simplification.

Look at the data before modelling it.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13.0, 3.8))
for bus in range(6):
    axes[0].plot(np.sort(Y[:, bus, 0]), lw=1.4, label=f"bus {bus}")
    axes[1].plot(np.sort(Y[:, bus, 1]), lw=1.4, label=f"bus {bus}")
axes[0].set_title("bus angle, sorted over the 800 cases")
axes[0].set_ylabel("theta [rad]")
axes[1].set_title("voltage deviation, sorted over the 800 cases")
axes[1].set_ylabel("V - 1 [p.u.]")
for a in axes:
    a.set_xlabel("case (sorted)"); a.grid(alpha=0.25)
    a.legend(frameon=False, fontsize=8, ncol=2)
plt.show()

print("per-bus standard deviation of theta [rad]:")
print(np.round(Y[:, :, 0].std(axis=0), 4))
print("per-bus standard deviation of V - 1 [p.u.]:")
print(np.round(Y[:, :, 1].std(axis=0), 4))
print("\nlargest angle difference across any line: %.1f degrees"
      % np.degrees(max(np.abs(Y[:, a, 0] - Y[:, b, 0]).max()
                       for a, b, _ in core.SIX_BUS_LINES)))

**What you should see.** Two fans of curves, per-bus standard deviations of
roughly `[0.0005 0.037 0.040 0.090 0.084 0.117]` for the angle and
`[0.013 0.014 0.013 0.011 0.008 0.010]` for the voltage, and a largest line
angle difference around 25 degrees.

Two features of that first row matter for the rest of the notebook.

**Bus 0 has no variation beyond the noise.** It is the angle reference: angles
are defined up to a common offset, so one bus is nominated and set to zero. A
permutation-equivariant model has no notion of bus number, so "the reference is
bus 0" is invisible to it. The reference must therefore be a **feature**:
`is_reference` is 1 at the reference bus and 0 elsewhere, and it permutes with
the buses. Anything positional must be a feature — the same point L12.1 makes
about measurement masks.

**The two target channels differ in scale by a factor of seven** — a factor of
fifty in variance. A plain mean squared error over both would weight angles
fifty times more than voltages without you choosing to. The fix here: divide
each channel by its standard deviation before training, convert back to
physical units before reporting.

---

## 2 · Splitting, and scaling the targets


In [ ]:
n_train = 600
Y_scaled = (Y / scale).astype(np.float32)

X_train, Y_train = X[:n_train], Y_scaled[:n_train]
X_test, Y_test = X[n_train:], Y_scaled[n_train:]
Y_test_physical = Y[n_train:]

A_hat = core.normalised_adjacency(A)
A_hat_t = torch.tensor(A_hat, dtype=torch.float32)

print("training cases :", X_train.shape[0])
print("held out cases :", X_test.shape[0])
print("scaled target standard deviation:",
      np.round(Y_train.reshape(-1, 2).std(axis=0), 3))
print("\nA_hat row sums :", np.round(A_hat.sum(axis=1), 3))

**What you should see.** 600 training cases, 200 held out, scaled target
standard deviations close to `[1. 1.]`, and the familiar
`[0.911 1.039 1.039 1.039 1.039 0.911]`.

The split is a plain first-600 / last-200 cut — safe here because the cases
are independent draws, and not safe on the time series in notebook 04.

---

## 3 · The layer

Notebook 02 built the Kipf and Welling convolution $\hat{A} H W$. Here you use
a small generalisation:

$$\mathbf{H}^{(l+1)} = \tanh\!\left(
\mathbf{H}^{(l)}\mathbf{W}_{\mathrm{self}} + \mathbf{b}
\;+\;\hat{\mathbf{A}}\,\mathbf{H}^{(l)}\mathbf{W}_{\mathrm{neigh}}\right)$$

A node's own state and its neighbours' aggregate get **separate** weight
matrices. In L5.1's message-passing form, this is $\psi$ a linear map and
$\phi$ an addition followed by $\tanh$. The separate self term matters because
an injection at bus $i$ enters the equation for $\theta_i$ differently from an
injection at a neighbour; it is worth about a factor of three in error here,
which you can check by editing one line.

Two construction choices, both course-wide habits:

* **`tanh`, not ReLU** — a ReLU network has zero second derivative almost
  everywhere, and Part 2's physics residuals need curvature (L4.1).
* **A residual connection after the first layer** — it makes depth trainable
  (L5.1), and on a graph it also gives each node a path to the output that is
  not averaged with its neighbours at every step: the antidote to the
  over-smoothing you watched in notebook 02.

### Your turn

Write the layer and the network.


In [ ]:
# TODO: build the message-passing layer and the network.
#
#   class GraphLayer(nn.Module):
#       def __init__(self, n_in, n_out):
#           super().__init__()
#           self.self_lin  = nn.Linear(n_in, n_out)                # own state
#           self.neigh_lin = nn.Linear(n_in, n_out, bias=False)    # neighbours
#
#       def forward(self, A_hat, H):
#           return self.self_lin(H) + A_hat @ self.neigh_lin(H)
#
#   class GraphNet(nn.Module):
#       def __init__(self, n_in=5, hidden=32, n_out=2, depth=3):
#           super().__init__()
#           self.layers = nn.ModuleList([
#               GraphLayer(n_in if k == 0 else hidden, hidden)
#               for k in range(depth)])
#           self.head = nn.Linear(hidden, n_out)
#
#       def forward(self, A_hat, H):
#           for k, layer in enumerate(self.layers):
#               Z = torch.tanh(layer(A_hat, H))
#               H = H + Z if k > 0 else Z          # residual after the first
#           return self.head(H)
#
# Note the shapes. H is (cases, 6, features) and A_hat is (6, 6); the product
# A_hat @ H broadcasts over the leading case axis, which is exactly what you
# want -- the same graph applied to every operating point.

raise NotImplementedError("Define GraphLayer and GraphNet")

In [ ]:
core.set_seed(0)
gnn = GraphNet(n_in=5, hidden=32, n_out=2, depth=3)
n_gnn = core.count_parameters(gnn)

with torch.no_grad():
    probe = gnn(A_hat_t, torch.tensor(X_train[:4]))
print("shape check:", tuple(torch.tensor(X_train[:4]).shape), "->", tuple(probe.shape))
print("parameters :", n_gnn)
for name, p in gnn.named_parameters():
    print(f"  {name:26s} {str(tuple(p.shape)):>12s}  {p.numel():5d}")

**What you should see.** `shape check: (4, 6, 5) -> (4, 6, 2)` and
`parameters : 4578`.

Six nodes in, six nodes out, and the node count appears nowhere in the parameter
list. Every weight matrix is indexed by *features*, never by *buses*. Add a
seventh bus and this model runs unchanged — which is the claim of the whole
architecture, and the reason L12.2 can talk about training on small cases and
deploying on larger ones.

---

## 4 · Training

Full batch, Adam, mean squared error on the scaled targets, 1500 epochs.
`core.train_graph` runs the loop for you; it is the same four lines as always,
with the adjacency passed through to the model.

### Your turn

In [ ]:
# TODO: train the graph network and report its error in physical units.
#
#   history_gnn = core.train_graph(gnn, A_hat, X_train, Y_train,
#                                  epochs=1500, lr=0.01,
#                                  X_val=X_test, Y_val=Y_test,
#                                  verbose_every=300)
#
# Then write a helper that predicts on a set and converts back to physical
# units, and use it to compute the per-channel root mean squared error:
#
#   def predict(model, A_hat_matrix, X_any):
#       model.eval()
#       with torch.no_grad():
#           out = model(torch.tensor(A_hat_matrix, dtype=torch.float32),
#                       torch.tensor(X_any))
#       return out.numpy() * scale          # back to radians and per unit
#
#   rmse_theta_gnn = sqrt(mean((pred[:, :, 0] - Y_test_physical[:, :, 0])**2))
#   rmse_volt_gnn  = ... the same for channel 1 ...

raise NotImplementedError("Train the graph network and compute its two RMSEs")

In [ ]:
print(f"\nangle   RMSE: {rmse_theta_gnn:.5f} rad  "
      f"({np.degrees(rmse_theta_gnn):.3f} degrees)")
print(f"voltage RMSE: {rmse_volt_gnn:.5f} p.u.")
print(f"measurement noise for comparison: {data['noise']:.5f}")

core.plot_curves(history_gnn, title="Graph network — 3 layers, 4578 parameters")
plt.show()

case = 0
fig, axes = plt.subplots(1, 2, figsize=(13.0, 3.8))
for ch, (name, unit) in enumerate([("angle", "rad"), ("voltage", "p.u.")]):
    axes[ch].plot(range(6), Y_test_physical[case, :, ch], "o-", ms=8,
                  color="#111111", label="truth")
    axes[ch].plot(range(6), pred_gnn[case, :, ch], "s--", ms=7,
                  color="#d94f2b", label="graph network")
    axes[ch].set_xticks(range(6))
    axes[ch].set_xticklabels(core.BUS_NAMES, rotation=30, ha="right", fontsize=8)
    axes[ch].set_ylabel(f"{name} [{unit}]")
    axes[ch].set_title(f"held-out case {case}: {name}")
    axes[ch].legend(frameon=False, fontsize=9); axes[ch].grid(alpha=0.25)
plt.show()

**What you should see.** A training curve falling smoothly, a validation
curve tracking it closely, and errors of roughly

```
angle   RMSE: 0.0043 rad  (0.25 degrees)
voltage RMSE: 0.0017 p.u.
```

Anything within a factor of two of these is a correct run.

In engineering units: a quarter of a degree, and 0.17 % of nominal voltage —
small against the spread of the data, several times the 0.0005 measurement
noise. The model has neither memorised the noise nor reached the floor.

There is almost no train/validation gap, despite 4578 parameters on 600 cases.
The model is *constrained*, not small: it cannot treat bus 3 differently from
bus 4 except through the graph and the features, so most ways of overfitting
are unavailable to it. Architecture as regularisation (L4.2).

---

## 5 · Two baselines, and an honest comparison

**The classical baseline.** The linearised DC power flow, $P = B\theta$: a
one-line matrix solve, no parameters, no data. If your network cannot beat it,
report that rather than hiding it.

**The architectural baseline.** A dense network on the flattened 30-vector,
predicting the flattened 12-vector — the model somebody would write who had
never heard of graph networks. Section 6 breaks it.

### Your turn


In [ ]:
# TODO: build both baselines.
#
# (a) the classical one -- no training involved:
#
#     theta_dc = core.dc_power_flow(data["P"][n_train:])
#     rmse_theta_dc = sqrt(mean((theta_dc - Y_test_physical[:, :, 0])**2))
#
#     Note it predicts angles only. The DC approximation says nothing at all
#     about voltage magnitude, which is worth remembering when comparing.
#
# (b) the dense one:
#
#     class DenseNet(nn.Module):
#         def __init__(self, hidden=64):
#             super().__init__()
#             self.net = nn.Sequential(nn.Linear(6 * 5, hidden), nn.Tanh(),
#                                      nn.Linear(hidden, hidden), nn.Tanh(),
#                                      nn.Linear(hidden, 6 * 2))
#         def forward(self, A_hat, H):          # A_hat accepted and ignored
#             flat = H.reshape(H.shape[0], -1)
#             return self.net(flat).reshape(H.shape[0], 6, 2)
#
#     Train it with the same call, the same epochs and the same learning rate,
#     record n_mlp = core.count_parameters(mlp), and compute rmse_theta_mlp and
#     rmse_volt_mlp the same way.
#
# The signature takes A_hat and ignores it, so the two models are
# interchangeable in every function below. That the dense model *can* ignore
# the graph is the point of section 6.

raise NotImplementedError("Build the DC baseline and the dense baseline")

In [ ]:
rows = [["DC power flow (no training)", "0", f"{rmse_theta_dc:.5f}", "not predicted"],
        ["graph network", f"{n_gnn:,}", f"{rmse_theta_gnn:.5f}",
         f"{rmse_volt_gnn:.5f}"],
        ["dense network", f"{n_mlp:,}", f"{rmse_theta_mlp:.5f}",
         f"{rmse_volt_mlp:.5f}"]]
print(core.error_table(rows, ["model", "parameters", "angle RMSE [rad]",
                              "voltage RMSE [p.u.]"]))

fig, ax = plt.subplots(figsize=(6.8, 4.2))
ax.plot(history_gnn["epoch"], history_gnn["val"], lw=1.7, color="#1f77b4",
        label=f"graph network ({n_gnn:,} par.)")
ax.plot(history_mlp["epoch"], history_mlp["val"], lw=1.7, color="#d94f2b",
        label=f"dense network ({n_mlp:,} par.)")
ax.set_yscale("log"); ax.set_xlabel("epoch")
ax.set_ylabel("validation MSE (scaled targets)")
ax.set_title("Held-out loss, same data and same recipe")
ax.legend(frameon=False, fontsize=9); ax.grid(alpha=0.25, which="both")
plt.show()

**What you should see.** Something close to

| model | parameters | angle RMSE [rad] | voltage RMSE [p.u.] |
| --- | --- | --- | --- |
| DC power flow (no training) | 0 | 0.00233 | not predicted |
| graph network | 4,578 | 0.0043 | 0.0017 |
| dense network | 6,924 | 0.0016 | 0.0007 |

**The dense network wins, by about a factor of three.** That is not a bug. On
thirty inputs with six hundred training cases, an unconstrained approximator
wins, and pretending otherwise would teach you to expect something that will
not happen in your own work.

**The DC power flow beats the graph network on angles.** Also expected: the
true angles are nearly linear in the injections, and the DC solve is exact for
the linear part with no fitting error. The graph network must learn the whole
map, including its linear part, from data.

**What the graph network buys** is not in this table:

* it predicts the voltage magnitude, which the DC solve does not;
* its parameter count does not grow with the network — the same 4,578 numbers
  describe six buses or six hundred;
* it cannot be broken by renumbering the buses.

The third is the subject of the next section. Note the caution from L12.1
slide 13: on a single fixed topology, "graph networks embed the topology" is
the *weakest* argument for them, because the topology is already known
exactly. Name what the architecture buys in *this* formulation.

---

## 6 · Permutation equivariance, demonstrated

$$f(\mathbf{P}\mathbf{X},\; \mathbf{P}\mathbf{A}\mathbf{P}^\top)
= \mathbf{P}\, f(\mathbf{X}, \mathbf{A})$$

Relabel the buses; the prediction must follow the relabelling. Notebook 02
verified this algebraically on one untrained layer. Now run it on a trained
model, and on the dense baseline — the experiment a sceptical grid operator
would ask for.

### Your turn


In [ ]:
# TODO: run the permutation test on both models.
#
#   perm = [3, 0, 5, 2, 4, 1]
#   Pm   = core.permutation_matrix(perm)
#
#   Permute the features of every held-out case and the adjacency matrix:
#       X_perm     = np.einsum("ij,njf->nif", Pm, X_test).astype(np.float32)
#       A_perm     = Pm @ A @ Pm.T
#       A_hat_perm = core.normalised_adjacency(A_perm)
#
#   For each model, compare
#       f(P X, P A P^T)          -- predict(model, A_hat_perm, X_perm)
#   with
#       P f(X, A)                -- np.einsum("ij,njf->nif", Pm, prediction)
#
#   Record, for each model, the largest absolute difference in the ANGLE
#   channel, in radians:  gap_gnn and gap_mlp.  Keep the permuted-input
#   predictions as pred_gnn_perm and pred_mlp_perm - the accuracy table in
#   the next cell reuses them.
#
# Feed the dense model the permuted adjacency too. It ignores it, which is the
# whole point -- it has no mechanism by which the relabelling could reach it.

raise NotImplementedError("Run the permutation test on the graph and dense models")

In [ ]:
print("permutation:", perm, " (the bus now called 0 used to be called 3)")
print()
print(f"graph network   largest |f(PX, PAP') - P f(X, A)| = {gap_gnn:.3e} rad")
print(f"dense network   largest |f(PX, PAP') - P f(X, A)| = {gap_mlp:.3e} rad")
print()
print(f"for scale, the spread of the angles themselves is "
      f"{Y_test_physical[:, :, 0].std():.4f} rad")

# accuracy after relabelling, in physical units
Y_test_perm = np.einsum("ij,njf->nif", Pm, Y_test_physical)
rmse_gnn_perm = float(np.sqrt(np.mean(
    (pred_gnn_perm[:, :, 0] - Y_test_perm[:, :, 0]) ** 2)))
rmse_mlp_perm = float(np.sqrt(np.mean(
    (pred_mlp_perm[:, :, 0] - Y_test_perm[:, :, 0]) ** 2)))
rows = [["graph network", f"{rmse_theta_gnn:.5f}", f"{rmse_gnn_perm:.5f}",
         f"{rmse_gnn_perm / rmse_theta_gnn:.1f} x"],
        ["dense network", f"{rmse_theta_mlp:.5f}", f"{rmse_mlp_perm:.5f}",
         f"{rmse_mlp_perm / rmse_theta_mlp:.1f} x"]]
print()
print(core.error_table(rows, ["model", "angle RMSE, original labels",
                              "angle RMSE, relabelled", "degradation"]))

case = 0
fig, ax = plt.subplots(figsize=(8.2, 4.0))
ax.plot(range(6), Y_test_perm[case, :, 0], "o-", ms=9, color="#111111",
        label="truth, relabelled")
ax.plot(range(6), pred_gnn_perm[case, :, 0], "s--", ms=7, color="#1f77b4",
        label="graph network on relabelled network")
ax.plot(range(6), pred_mlp_perm[case, :, 0], "^--", ms=7, color="#d94f2b",
        label="dense network on relabelled network")
ax.set_xticks(range(6)); ax.set_xlabel("bus, new numbering")
ax.set_ylabel("theta [rad]")
ax.set_title("The same six buses, renumbered")
ax.legend(frameon=False, fontsize=9); ax.grid(alpha=0.25)
plt.show()

**What you should see.** A gap of order $10^{-7}$ radians for the graph
network — float32 rounding, exactly zero in exact arithmetic — against a gap of
order $10^{-1}$ radians for the dense network, comparable to the entire spread
of the data. The degradation table shows the graph network unchanged to five
decimal places, and the dense network worse than predicting the mean.

Three things to take from this.

**The graph network's equivariance is exact and was never trained for.** Every
operation is either per-node with shared weights or a sum over neighbours;
neither can see a bus number. That is an architectural guarantee, as against a
learned tendency.

**The dense network cannot have it.** Its first-layer weights are indexed by
position, so permuting the input permutes which weight each number meets. More
data would only make it approximately equivariant over the permutations it saw
— 720 for six buses, $n!$ in general.

**This is why the property is worth accuracy.** The dense network's factor of
three depends on a convention: the order somebody listed the buses in a file.
Change the convention and it fails silently. A model whose correctness depends
on the row order of a CSV cannot be handed to an operator.

L12.2 puts it in one sentence: *the model has no notion of bus number, only of
structure.*

---

## 7 · How deep? The graph answers

Notebook 02 measured this network's diameter: **three hops**. A
message-passing layer moves information one hop, so fewer than three layers
makes it structurally impossible for bus 0's injection to influence bus 5's
angle — and in a power system it does. Depth is a physical quantity here, not
a hyperparameter to tune blindly (L12.2 makes the same point).

### Your turn

Sweep the depth from one to four and plot the held-out error against it.


In [ ]:
# TODO: train a GraphNet at each depth in (1, 2, 4, 8, 12) and record
# the held-out angle RMSE in radians and the parameter count.
#
#   depth_results = []          # (depth, n_parameters, rmse_theta, rmse_volt)
#   for depth in (1, 2, 4, 8, 12):
#       core.set_seed(0)
#       model = GraphNet(depth=depth)
#       core.train_graph(model, A_hat, X_train, Y_train, epochs=1500, lr=0.01)
#       ... predict, convert to physical units, compute both RMSEs ...
#
# Use the same seed for every depth, so that the only difference is the depth.
# This takes four or five minutes; the twelve-layer model is the slow one.

raise NotImplementedError("Sweep the depth from one to twelve")

In [ ]:
rows = [[d, f"{n:,}", f"{rt:.5f}", f"{rv:.5f}"]
        for d, n, rt, rv in depth_results]
print(core.error_table(rows, ["layers", "parameters", "angle RMSE [rad]",
                              "voltage RMSE [p.u.]"]))

fig, ax = plt.subplots(figsize=(6.6, 4.0))
ax.plot([r[0] for r in depth_results], [r[2] for r in depth_results],
        "o-", lw=1.8, ms=8, color="#1f77b4", label="angle")
ax.plot([r[0] for r in depth_results], [r[3] for r in depth_results],
        "s-", lw=1.8, ms=8, color="#0f9d58", label="voltage")
ax.axhline(rmse_theta_dc, color="#d94f2b", ls="--", lw=1.4,
           label="DC power flow, angle")
ax.axvline(3, color="#999999", ls=":", lw=1.4)
ax.text(3.03, ax.get_ylim()[1] * 0.7, "graph diameter", fontsize=9,
        color="#555555")
ax.set_yscale("log"); ax.set_xticks([1, 2, 4, 8, 12])
ax.set_xlabel("message-passing layers")
ax.set_ylabel("held-out RMSE (physical units)")
ax.set_title("Depth is a statement about the graph")
ax.legend(frameon=False, fontsize=9); ax.grid(alpha=0.25, which="both")
plt.show()

**What you should see.** Something close to

| layers | parameters | angle RMSE [rad] | voltage RMSE [p.u.] |
| --- | --- | --- | --- |
| 1 | 418 | 0.0304 | 0.0072 |
| 2 | 2,498 | 0.0084 | 0.0031 |
| 4 | 6,658 | 0.0040 | 0.0009 |
| 8 | 14,978 | 0.0045 | 0.0010 |
| 12 | 23,298 | 0.0081 | 0.0030 |

The angle error falls steeply up to the graph diameter — three — and the
returns run out just past it: four layers is the best, eight is no better,
and twelve is worse. That turn is **over-smoothing** (L5.1): every
message-passing layer averages over neighbours, and enough averaging pulls
all six node representations towards the same value. It arrives late here
because `GraphNet` adds each layer's output to its input — the residual
connection from L5.1 slide 10 doing on a graph exactly what it does in a deep
image network. Remove the residual (`H = Z` for every layer) and rerun the
twelve-layer model: on our runs the angle error jumps from 0.0081 to 0.027,
worse than a single layer. Depth on a graph needs the same rescue depth in an
image network needs.

One qualification makes that honest: depth was not the only thing that
changed, since a deeper network also has more parameters. What the diameter
argument predicts is the *shape* of the curve — large gains up to three, little
after, then losses — and that is what you see. A cleaner experiment holds the parameter
count fixed by shrinking the width as depth grows (Ex_04's budget); notebook 05
suggests it as an extension.

The voltage channel improves slightly at four layers, consistent with the
physics: the sag term depends on every line's loading, so voltage is a more
global quantity than a bus angle.

---

## 8 · One thing the graph network can do that the dense one cannot

A line trips: seven lines instead of eight, a different graph, and the new
admittances are known exactly from line data — the graph network does not learn
them, it **consumes** them (L12.2). Hand it the new $\hat{A}$. The dense
network's input is a 30-vector with no room for a topology in it, so it cannot
be told at all.

### Your turn


In [ ]:
# TODO: trip line 3 -- the line from bus 1 to bus 3 -- and test both models
# on the new network WITHOUT retraining either of them.
#
#   lines_trip = core.lines_without(3)
#   trip = core.six_bus_dataset(n_cases=200, seed=77, lines=lines_trip)
#   A_trip     = trip["A"]
#   A_hat_trip = core.normalised_adjacency(A_trip)
#
#   graph network : predict(gnn, A_hat_trip, trip["X"])   -- told about the trip
#   dense network : predict(mlp, A_hat,      trip["X"])   -- cannot be told
#
# Compute the angle RMSE of each against trip["Y"][:, :, 0], as
# rmse_theta_gnn_trip and rmse_theta_mlp_trip.

raise NotImplementedError("Test both models on the tripped network")

In [ ]:
spread = float(trip["Y"][:, :, 0].std())
rows = [["graph network (given the new A)", f"{rmse_theta_gnn:.5f}",
         f"{rmse_theta_gnn_trip:.5f}"],
        ["dense network (cannot be told)", f"{rmse_theta_mlp:.5f}",
         f"{rmse_theta_mlp_trip:.5f}"],
        ["predicting the mean", "-", f"{spread:.5f}"]]
print(core.error_table(rows, ["model", "angle RMSE, intact",
                              "angle RMSE, line 1-3 tripped"]))

fig, axes = plt.subplots(1, 2, figsize=(13.0, 3.8))
core.plot_graph(A, ax=axes[0], title="intact: 8 lines")
core.plot_graph(A_trip, ax=axes[1], title="line 1-3 tripped: 7 lines")
plt.show()

**What you should see.** Both models get much worse, and the graph network
gets less worse — something like

| model | angle RMSE, intact | angle RMSE, line 1-3 tripped |
| --- | --- | --- |
| graph network (given the new A) | 0.0043 | 0.115 |
| dense network (cannot be told) | 0.0016 | 0.265 |

against a spread in the data of about 0.32 radians.

Do not oversell this. Twice as good on a network it never saw is a real
difference, caused entirely by consuming the new topology — and 0.115 radians
is still not a usable prediction. An operator would throw it away.

The model was trained on one topology, so it learned that topology's
statistics along with the physics. Accepting a new graph is **necessary** for
transfer, not **sufficient**. Sufficient would be training across a mixture of
topologies — the intact network plus a set of contingencies — so the only thing
the weights can encode is the local rule. That reformulation is what L12.1
slide 13 reaches, and the dense network has no version of it at any amount of
data, because there is nowhere to put the topology.

---

## 9 · Save

Notebook 05 reads this file.


In [ ]:
os.makedirs(core.OUTPUT_DIR, exist_ok=True)
path = os.path.join(core.OUTPUT_DIR, "nb03_gnn.npz")
np.savez(path,
         n_gnn=n_gnn, n_mlp=n_mlp,
         rmse_theta_gnn=rmse_theta_gnn, rmse_volt_gnn=rmse_volt_gnn,
         rmse_theta_mlp=rmse_theta_mlp, rmse_volt_mlp=rmse_volt_mlp,
         rmse_theta_dc=rmse_theta_dc,
         gap_gnn=gap_gnn, gap_mlp=gap_mlp,
         rmse_gnn_perm=rmse_gnn_perm, rmse_mlp_perm=rmse_mlp_perm,
         rmse_theta_gnn_trip=rmse_theta_gnn_trip,
         rmse_theta_mlp_trip=rmse_theta_mlp_trip,
         depth_results=np.asarray(depth_results, dtype=float),
         val_gnn=history_gnn["val"], val_mlp=history_mlp["val"])
print("wrote", path)

**What you should see.** `wrote .../Ex05_outputs/nb03_gnn.npz`.

---

## 10 · Before you move on

Answer these here. Question 2 is the one notebook 05's report is built around.

1. The `is_reference` feature exists because a permutation-equivariant model has
   no notion of bus number. Name one other quantity in a power system that you
   would have to supply as a feature rather than as a convention, and say what
   goes wrong if you do not.
2. The dense network was three times more accurate and completely broken by a
   relabelling. **Which would you deploy, and what would have to be true about
   the deployment for that to be the right choice?**
3. The depth sweep flattened at three layers, which is the diameter of this
   graph. Given a 300-bus network with a diameter of twelve, what would you
   conclude about how deep the model should be — and what would stop you simply
   using twelve layers?
4. The DC power flow beat the graph network on angles, using no data and no
   parameters. Under what change to this problem would that stop being true?

---

Continue with **`Ex05_04_sequence_model.ipynb`**, which leaves graphs for
sequences — the second half of lecture block L5.

*Write your answers here.*

1.
2.
3.
4.